In [ ]:
import hashlib
import os
import csv
import glob
import multiprocessing
from tqdm import tqdm
import pandas as pd
import sys
csv.field_size_limit(sys.maxsize)

# get number of available CPUs for multiprocessing
NUM_PROCESSES = multiprocessing.cpu_count() # Be careful, more CPUS can lead to memory errors
# NUM_PROCESSES = 32
print(f"Number of CPUs: {NUM_PROCESSES}")

FILES_TO_SCAN = glob.glob("./**/*.txt", recursive=True)
OUTPUT_DIR = "z-code/xxx-output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DOCS_TO_EXCLUDE = glob.glob("./**/README*.*", recursive=True) + \
                  glob.glob("./**/deploy_spacy_*.*", recursive=True) + \
                  glob.glob("./**/*checkpoint*", recursive=True)


HASH_PARTS_OUTPUT_DIRS = [f"{OUTPUT_DIR}/hash_parts/output_part_{i}" for i in range(NUM_PROCESSES)]
# Ensure output directories exist
for output_dir in HASH_PARTS_OUTPUT_DIRS:
    os.makedirs(output_dir, exist_ok=True)
    
HASHES_CSV = "final_hashes.csv"

CSV_PARTS_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "csv_parts")
os.makedirs(CSV_PARTS_OUTPUT_DIR, exist_ok=True)


def calculate_hash(file_path):
    """Calculate the SHA-256 hash of a file."""
    hash_func = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            hash_func.update(chunk)
    return file_path, hash_func.hexdigest()

def process_files(file_list, output_csv):
    """Process a list of files, calculate hashes, and save to a CSV."""
    results = []

    # for file_path in tqdm(file_list, desc=f"Processing {len(file_list)} files", position=0):
    #     results.append(calculate_hash(file_path))
    for file_path in file_list:
        results.append(calculate_hash(file_path))
    
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Filepath", "Hash"])
        writer.writerows(results)


def parallel_scan(files_or_pattern):
    """Parallelized function to scan files, distribute tasks, and merge results."""
    
    print("Scanning files in parallel...")
    
    if isinstance(files_or_pattern, str):
        all_files = glob.glob(files_or_pattern, recursive=True)
    else:
        all_files = files_or_pattern
        
    total_files = len(all_files)

    # Split files into chunks for parallel processing
    chunk_size = total_files // NUM_PROCESSES
    file_chunks = [all_files[i:i + chunk_size] for i in range(0, total_files, chunk_size)]

    # Adjust for rounding errors
    while len(file_chunks) < NUM_PROCESSES:
        file_chunks.append([])

    # Define output CSVs for each process
    output_csvs = [os.path.join(HASH_PARTS_OUTPUT_DIRS[i], f"hashes_part_{i}.csv") for i in range(NUM_PROCESSES)]

    # Run processes in parallel
    with multiprocessing.Pool(NUM_PROCESSES) as pool:
        pool.starmap(process_files, zip(file_chunks, output_csvs))

    print("All individual CSVs generated. Merging into final CSV...")

    # Merge all CSVs into one
    merged_df = pd.concat([pd.read_csv(csv_file) for csv_file in output_csvs], ignore_index=True)
    merged_df.to_csv("final_hashes.csv", index=False)

    print("Final merged CSV saved as: final_hashes.csv")

131072

In [14]:
import hashlib
import glob

TEST_PATH = "nbs/preprocessor/test_data/"

class HashExtractor:
    CSV_COLUMNS = ["Filepath", "Hash"]
    
    def get_file_hash(self, file_path):
        """Calculate the SHA-256 hash of a file."""
        hash_func = hashlib.sha256()
        with open(file_path, "rb") as f:
            while chunk := f.read(8192):
                hash_func.update(chunk)
        return file_path, hash_func.hexdigest()
    
    def get_batch_hash(self, file_list, output_csv=None, progress_bar=True):
        """Process a list of files, calculate hashes, and save to a CSV."""
        ls_hashes = []

        if progress_bar:
            file_list = tqdm(file_list, desc=f"Processing {len(file_list)} files", position=0)

        for file_path in file_list:
                ls_hashes.append(self.get_file_hash(file_path))
        
        if output_csv is not None:
            with open(output_csv, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(self.CSV_COLUMNS)
                writer.writerows(ls_hashes)
        
        return ls_hashes

ls_files = glob.glob("nbs/preprocessor/test_data/test*/**/*.txt", recursive=True)
print(ls_files)
he = HashExtractor()
print(he.get_file_hash("nbs/preprocessor/test_data/test1/test11/example1.txt"))
print(he.get_batch_hash(ls_files, os.path.join(TEST_PATH, "hashes.csv")))

['nbs/preprocessor/test_data/test2/example2.txt', 'nbs/preprocessor/test_data/test2/test21/example21.txt', 'nbs/preprocessor/test_data/test1/test11/example1.txt']
('nbs/preprocessor/test_data/test1/test11/example1.txt', '96e919bedc30d13e853fc5ad04cd44275496666eaf0a01d651f03b61a221eb77')


Processing 3 files: 100%|██████████| 3/3 [00:00<00:00, 9761.76it/s]

[('nbs/preprocessor/test_data/test2/example2.txt', '305ecb9a16de7a26321561efb8ee442fa105e3137457361d89715c2db3c6783f'), ('nbs/preprocessor/test_data/test2/test21/example21.txt', '96e919bedc30d13e853fc5ad04cd44275496666eaf0a01d651f03b61a221eb77'), ('nbs/preprocessor/test_data/test1/test11/example1.txt', '96e919bedc30d13e853fc5ad04cd44275496666eaf0a01d651f03b61a221eb77')]


In [76]:
class ParallelHashExtractor:
    
    def __init__(self, output_dir, num_processes=None):
        self.num_processes = num_processes if num_processes is not None else multiprocessing.cpu_count()
        self.output_dir = output_dir
        self.hash_parts_output_dirs = [f"{output_dir}/hash_parts/output_part_{i}" for i in range(num_processes)]
        self.csv_parts_output_dir = os.path.join(output_dir, "csv_parts")
        self.output_csv = os.path.join(output_dir, "hashes.csv")
        os.makedirs(self.csv_parts_output_dir, exist_ok=True)
        
        for output_dir in self.hash_parts_output_dirs:
            os.makedirs(output_dir, exist_ok=True)
        
    def get_batch_hash(self, files_or_pattern, progress_bar=False, output_csv=None):
        """Parallelized function to scan files, distribute tasks, and merge results."""

        he = HashExtractor()
        self.CSV_COLUMNS = he.CSV_COLUMNS
        
        output_csv = output_csv if output_csv is not None else self.output_csv
        
        print(f"Scanning files in parallel on {self.num_processes} cores...")

        if isinstance(files_or_pattern, str):
            all_files = glob.glob(files_or_pattern, recursive=True)
        else:
            all_files = files_or_pattern

        total_files = len(all_files)

        # Calculate the chunk size
        chunk_size = total_files // self.num_processes
        remainder = total_files % self.num_processes

        # Create the file chunks
        file_chunks = []
        start_idx = 0

        for i in range(self.num_processes):
            # Distribute the remainder files among the chunks
            end_idx = start_idx + chunk_size + (1 if i < remainder else 0)
            file_chunks.append(all_files[start_idx:end_idx])
            start_idx = end_idx
            
        print(file_chunks)
        # Define output CSVs for each process
        output_csvs = [os.path.join(self.hash_parts_output_dirs[i], f"hashes_part_{i}.csv") for i in range(self.num_processes)]
        
        # Run processes in parallel
        from functools import partial
        partial_get_batch_hash = partial(he.get_batch_hash, progress_bar=progress_bar)
        
        with multiprocessing.Pool(self.num_processes) as pool:
            pool.starmap(partial_get_batch_hash, zip(file_chunks, output_csvs))

        print("All individual CSVs generated. Merging into final CSV...")

        # Merge all CSVs into one
        merged_df = pd.concat([pd.read_csv(csv_file) for csv_file in output_csvs], ignore_index=True)
        merged_df.to_csv(self.output_csv, index=False)

        print(f"Final merged CSV saved as: {self.output_csv}")
        
        return merged_df
        
phe = ParallelHashExtractor("nbs/preprocessor/", num_processes=4)
phe.get_batch_hash("nbs/preprocessor/test_data/test*/**/*.txt");

Scanning files in parallel on 4 cores...
[['nbs/preprocessor/test_data/test2/example2.txt'], ['nbs/preprocessor/test_data/test2/test21/example21.txt'], ['nbs/preprocessor/test_data/test1/test11/example1.txt'], []]
All individual CSVs generated. Merging into final CSV...
Final merged CSV saved as: nbs/preprocessor/hashes.csv


In [58]:
class CSVConcatenator:
    
    CSV_COLUMNS = ["filepath", "text"]
    
    def __init__(self, output_dir, num_processes=None):
        self.num_processes = num_processes if num_processes is not None else multiprocessing.cpu_count()
        self.output_dir = output_dir
        self.csv_parts_output_dir = os.path.join(output_dir, "csv_parts")
        os.makedirs(self.csv_parts_output_dir, exist_ok=True)
        
 
    def get_file_content(self, file_path):
        """Reads a file and returns its name and content (single line)."""
        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read().replace("\n", " ")
            return file_path, content
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            return file_path, ""
        
    def get_batch_content(self, file_list, output_csv=None, progress_bar=True):
        """Process a list of files, read contents, and save to a CSV."""
        ls_contents = []

        if progress_bar:
            file_list = tqdm(file_list, desc=f"Processing {len(file_list)} files", position=0)

        for file_path in file_list:
            ls_contents.append(self.get_file_content(file_path))
        
        if output_csv is not None:
            with open(output_csv, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(self.CSV_COLUMNS)
                writer.writerows(ls_contents)
        
        return ls_contents
        


csvc = CSVConcatenator("nbs/preprocessor/", num_processes=2)
print(csvc.get_file_content("nbs/preprocessor/test_data/test1/test11/example1.txt"))    
print(csvc.get_batch_content(ls_files, os.path.join(TEST_PATH, "contents.csv")))

('nbs/preprocessor/test_data/test1/test11/example1.txt', 'Hey there!')


Processing 3 files: 100%|██████████| 3/3 [00:00<00:00, 9649.47it/s]

[('nbs/preprocessor/test_data/test2/example2.txt', 'Sanlúcar de Barameda, Cádiz'), ('nbs/preprocessor/test_data/test2/test21/example21.txt', 'Hey there!'), ('nbs/preprocessor/test_data/test1/test11/example1.txt', 'Hey there!')]


In [67]:
class ParallelCSVConcatenator:
    
    def __init__(self, output_dir, num_processes=None):
        self.num_processes = num_processes if num_processes is not None else multiprocessing.cpu_count()
        self.output_dir = output_dir
        self.csv_parts_output_dir = os.path.join(output_dir, "csv_parts")
        os.makedirs(self.csv_parts_output_dir, exist_ok=True)
    
    def get_batch_content(self, files_or_pattern, progress_bar=False, output_csv=None):
        
        csvc = CSVConcatenator(self.output_dir, num_processes=self.num_processes)
        self.CSV_COLUMNS = csvc.CSV_COLUMNS
        
        output_csv = output_csv if output_csv is not None else os.path.join(self.output_dir, "contents.csv")
        
        print(f"Scanning files in parallel on {self.num_processes} cores...")

        if isinstance(files_or_pattern, str):
            all_files = glob.glob(files_or_pattern, recursive=True)
        else:
            all_files = files_or_pattern

        total_files = len(all_files)

        # Calculate the chunk size
        chunk_size = total_files // self.num_processes
        remainder = total_files % self.num_processes

        # Create the file chunks
        file_chunks = []
        start_idx = 0

        for i in range(self.num_processes):
            # Distribute the remainder files among the chunks
            end_idx = start_idx + chunk_size + (1 if i < remainder else 0)
            file_chunks.append(all_files[start_idx:end_idx])
            start_idx = end_idx
            
        print(file_chunks)
        # Define output CSVs for each process
        output_csvs = [os.path.join(self.csv_parts_output_dir, f"contents_part_{i}.csv") for i in range(self.num_processes)]
        
        # Run processes in parallel
        from functools import partial
        partial_get_batch_content = partial(csvc.get_batch_content, progress_bar=progress_bar)
        
        with multiprocessing.Pool(self.num_processes) as pool:
            pool.starmap(partial_get_batch_content, zip(file_chunks, output_csvs))

        print("All individual CSVs generated. Merging into final CSV...")

        # Merge all CSVs into one
        merged_df = pd.concat([pd.read_csv(csv_file) for csv_file in output_csvs], ignore_index=True)
        merged_df.to_csv(output_csv, index=False)

        print(f"Final merged CSV saved as: {output_csv}")
        
        
pcsv = ParallelCSVConcatenator("nbs/preprocessor/", num_processes=2)
pcsv.get_batch_content("nbs/preprocessor/test_data/test*/**/*.txt")

Scanning files in parallel on 2 cores...
[['nbs/preprocessor/test_data/test2/example2.txt', 'nbs/preprocessor/test_data/test2/test21/example21.txt'], ['nbs/preprocessor/test_data/test1/test11/example1.txt']]
All individual CSVs generated. Merging into final CSV...
Final merged CSV saved as: nbs/preprocessor/contents.csv


In [81]:
class HashDeduplicator:
    """Apply deduplication based on hashes. If specified, uses a CSV with hashes."""
    
    def __init__(self, files_or_pattern, output_dir, hashes_csv=None, num_processes=None):
        self.files_or_pattern = files_or_pattern
        self.output_dir = output_dir
        self.hashes_csv = hashes_csv
        self.df_hashes = pd.read_csv(hashes_csv) if hashes_csv is not None else None
        self.num_processes = num_processes if num_processes is not None else multiprocessing.cpu_count()
        
    def deduplicate_files(self, file_list):
        """Deduplicate files based on hashes."""
        if self.df_hashes is not None:
            # Use the hashes from the CSV
            df_hashes = self.df_hashes
        else:
            # Calculate the hashes on the fly
            phe = ParallelHashExtractor(self.output_dir, num_processes=self.num_processes)
            hashes = phe.get_batch_hash(file_list, progress_bar=True)
            df_hashes = pd.DataFrame(hashes, columns=phe.CSV_COLUMNS)
        
        # Deduplicate based on hashes
        df_dedup = df_hashes.drop_duplicates(subset=phe.CSV_COLUMNS[1], keep="first")[phe.CSV_COLUMNS[0]].tolist()
        # print(f"Original number of files: {len(df_hashes)}")
        # print(f"Number of deduplicated files: {len(df_dedup)}")
        return df_dedup
    
    def get_deduplicated_files(self, output_csv=None):
        """Get the deduplicated files."""
        
        pcsv = ParallelCSVConcatenator(self.output_dir, num_processes=self.num_processes)
        ls_files = self.deduplicate_files(self.files_or_pattern)
        return pcsv.get_batch_content(ls_files, output_csv=output_csv)
    
    
hd = HashDeduplicator("nbs/preprocessor/test_data/test*/**/*.txt", "nbs/preprocessor/", num_processes=2)

hd.get_deduplicated_files("nbs/preprocessor/deduplicated_contents.csv");
# hd.deduplicate_files(ls_files);

Scanning files in parallel on 2 cores...
[['nbs/preprocessor/test_data/test2/example2.txt', 'nbs/preprocessor/test_data/test2/test21/example21.txt'], ['nbs/preprocessor/test_data/test1/test11/example1.txt']]


Processing 1 files: 100%|██████████| 1/1 [00:00<00:00, 669.27it/s]


All individual CSVs generated. Merging into final CSV...
Final merged CSV saved as: nbs/preprocessor/hashes.csv
Scanning files in parallel on 2 cores...
[['nbs/preprocessor/test_data/test2/example2.txt'], ['nbs/preprocessor/test_data/test2/test21/example21.txt']]
All individual CSVs generated. Merging into final CSV...
Final merged CSV saved as: nbs/preprocessor/deduplicated_contents.csv
